<a href="https://colab.research.google.com/github/Shukti-cyber/ML/blob/main/Image%20dector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== KernelVision: Handwritten Digit Recognizer - paste this whole cell into Colab and press play =====
import subprocess, re, time, os

subprocess.run("pip -q install streamlit", shell=True)

APP_CODE = r'''
import numpy as np
import streamlit as st
from PIL import Image, ImageOps
from scipy import ndimage as ndi
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

st.set_page_config(page_title="Digit Recognizer", page_icon="🔷", layout="centered")
st.markdown("""<style>
.big{background:#0f2340;border:1px solid #1e3a5f;border-left:5px solid #22d3ee;
border-radius:10px;padding:18px 22px;text-align:center}
.big .l{color:#8fb3d9;font-size:.8rem;letter-spacing:.08em;text-transform:uppercase}
.big .v{color:#e6edf7;font-size:4rem;font-weight:800;line-height:1.1}
</style>""", unsafe_allow_html=True)


# ---------------- BACKEND 1: train the model (runs once, then cached) ----------------
@st.cache_resource
def train_model():
    digits = load_digits()                                   # 1797 images, 8x8, labels 0-9
    X, y = digits.data, digits.target
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    model = Pipeline([("scaler", StandardScaler()),          # scale pixels
                      ("svm", SVC(kernel="rbf", C=10, gamma="scale"))])   # SVM with RBF kernel
    model.fit(X_tr, y_tr)                                    # learn from training data only
    acc = accuracy_score(y_te, model.predict(X_te))          # test on unseen data
    return model, acc


# ---------------- BACKEND 2: turn any photo into an 8x8 digit image ----------------
def preprocess(img):
    im = ImageOps.exif_transpose(img).convert("L")           # 1. grayscale
    im.thumbnail((400, 400))
    g = ndi.gaussian_filter(np.asarray(im, dtype=float), 1.0)          # 2. smooth noise
    bg = ndi.gaussian_filter(g, sigma=max(g.shape) / 8)                # 3. estimate paper / lighting
    med = np.median(g)
    dark_ink = (med - np.percentile(g, 1)) >= (np.percentile(g, 99) - med)
    ink = np.clip((bg - g) if dark_ink else (g - bg), 0, None)         # 4. ink = difference from paper
    if ink.max() <= 0:
        return None
    mask = ink > 0.35 * np.percentile(ink, 99.5)
    H, W = mask.shape
    lab, _ = ndi.label(ndi.binary_dilation(mask, iterations=4))        # 5. group strokes into blobs
    best, best_score = 0, 0.0
    for i, sl in enumerate(ndi.find_objects(lab), start=1):
        h, w = sl[0].stop - sl[0].start, sl[1].stop - sl[1].start
        if min(h, w) < 0.08 * max(h, w) and max(h, w) > 0.5 * max(H, W):
            continue                                                   # skip long thin ruled lines
        score = ink[lab == i].sum()
        if score > best_score:
            best, best_score = i, score
    if best == 0:
        return None
    ink = ink * ((lab == best) & ndi.binary_dilation(mask, iterations=1))   # 6. keep the digit only
    ys, xs = np.where(ink > 0)
    crop = ink[ys.min():ys.max() + 1, xs.min():xs.max() + 1]           # 7. crop around the digit
    h, w = crop.shape
    side = int(max(h, w) * 1.1)
    c = np.zeros((side, side))
    oy, ox = (side - h) // 2, (side - w) // 2
    c[oy:oy + h, ox:ox + w] = crop                                     # 8. centre on a square canvas
    for _ in range(40):                                                # 9. thicken thin pencil strokes
        if (c > 0.35 * c.max()).mean() >= 0.15:
            break
        c = ndi.grey_dilation(c, size=(3, 3))
    c = c / c.max() * 255
    a = np.asarray(Image.fromarray(c.astype(np.uint8)).resize((8, 8), Image.BOX), dtype=float)
    return (a / a.max() * 16).ravel()                                  # 10. resize to 8x8, scale 0-16


# ---------------- FRONTEND: what the user sees ----------------
model, acc = train_model()
st.title("🔷 Handwritten Digit Recognizer")
st.caption(f"SVM (RBF kernel) trained on the scikit-learn Digits dataset  •  test accuracy: {acc:.1%}")

file = st.file_uploader("Upload an image of ONE handwritten digit (PNG / JPG)", type=["png", "jpg", "jpeg"])
if file is not None:
    img = Image.open(file)
    vec = preprocess(img)
    if vec is None:
        st.error("Could not find a digit in this image. Try a thicker, darker digit on plain paper.")
    else:
        pred = int(model.predict(vec.reshape(1, -1))[0])
        c1, c2, c3 = st.columns(3)
        with c1:
            st.markdown("**Uploaded image**")
            st.image(img, use_container_width=True)
        with c2:
            st.markdown("**What the model sees (8×8)**")
            small = Image.fromarray((vec.reshape(8, 8) / 16 * 255).astype(np.uint8))
            st.image(small.resize((200, 200), Image.NEAREST), use_container_width=True)
        with c3:
            st.markdown(f'<div class="big"><div class="l">Predicted digit</div><div class="v">{pred}</div></div>',
                        unsafe_allow_html=True)
        st.caption("Photos are shrunk to 8×8 pixels, so results are approximate. "
                   "Best with one thick, dark digit on plain paper.")
'''

open("app.py", "w").write(APP_CODE)                      # 1. save the website code as app.py
subprocess.run("pkill -f '[s]treamlit run'; pkill -f '[c]loudflared tunnel'", shell=True)
if not os.path.exists("cloudflared"):                    # 2. download the tunnel tool (gives a public link)
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True)

subprocess.Popen("streamlit run app.py --server.headless true --server.port 8501 "
                 "--theme.base dark --theme.primaryColor '#22d3ee' --theme.backgroundColor '#0a1628' "
                 "--theme.secondaryBackgroundColor '#0f2340' > streamlit.log 2>&1", shell=True)   # 3. start the website
time.sleep(8)
tunnel = subprocess.Popen("./cloudflared tunnel --url http://localhost:8501", shell=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)             # 4. make the public link
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\nOPEN YOUR WEBSITE HERE  ->", m.group(0))
        break


OPEN YOUR WEBSITE HERE  -> https://bathrooms-afford-tribute-scope.trycloudflare.com
